# 06 — Full Research Workflow

## Objective

Connect the agent core (notebook 04) to the Evaluator-Optimizer workflow (notebook 05) so the
system runs as one flow for a stock ticker:

**plan → route → specialist agents → synthesis → draft → evaluate → refine → memory**

In notebook 05, the evaluator was tested on sample drafts written in the notebook. Here, the
draft comes from the orchestrator's actual output, so the evaluator is grading the system's own
analysis.

The connecting code is in `src/research_pipeline.py`:

| Function | Purpose |
|---|---|
| `build_research_draft(result)` | Turns the `orchestrate_research()` output into a written draft |
| `build_raw_evidence(ticker)` | Collects the evidence the evaluator checks the draft against (same format as notebook 05) |
| `run_full_research(ticker)` | Runs orchestration, then the evaluator-optimizer loop with memory |

## 1. Setup

This notebook needs the processed data from notebooks 01-03 and the LLM settings used in
notebook 05 (`LLM_API_KEY`, `LLM_MODEL`, and `LLM_BASE_URL` if not using OpenAI).

The `.env` file is loaded before the evaluator is imported, because the evaluator and optimizer
read the LLM settings at import time.

In [1]:
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

load_dotenv(PROJECT_ROOT / ".env")

if not os.getenv("LLM_API_KEY"):
    raise ValueError("LLM_API_KEY is not set. Add it to .env before running this notebook.")

from src.memory import ResearchMemoryStore
from src.orchestrator import display_orchestration, orchestrate_research
from src.research_pipeline import (
    build_raw_evidence,
    build_research_draft,
    run_full_research,
)

pd.set_option("display.max_colwidth", 120)

print(f"Project root: {PROJECT_ROOT}")
print(f"LLM model: {os.getenv('LLM_MODEL', 'gpt-4o-mini (default)')}")

Project root: c:\Users\cotto\Investment-Research-Multi-Agent-System
LLM model: gpt-4o-mini


## 2. Check Required Inputs

In [2]:
PROCESSED = PROJECT_ROOT / "data" / "processed"

REQUIRED_FILES = [
    "news_research_results.csv",
    "market_data_clean.csv",
    "sec_form4_clean.csv",
    "daily_evidence.csv",
]

missing = [name for name in REQUIRED_FILES if not (PROCESSED / name).exists()]

if missing:
    raise FileNotFoundError(
        "Missing processed inputs. Run notebooks 00-03 first:\n" + "\n".join(missing)
    )

for name in REQUIRED_FILES:
    print(f"{name:<28} {len(pd.read_csv(PROCESSED / name)):>6} rows")

TICKER = "AAPL"
MEMORY_PATH = PROCESSED / "memory_store.json"

news_research_results.csv         3 rows
market_data_clean.csv            44 rows
sec_form4_clean.csv              82 rows
daily_evidence.csv               58 rows


## 3. Step 1: Plan, Route, and Run Specialist Agents

The orchestrator from notebook 04 plans the research, routes each step to a specialist, and
combines the results into risks and catalysts. This step does not use the LLM.

In [3]:
orchestration = orchestrate_research(TICKER, PROJECT_ROOT)
display_orchestration(orchestration)

Ticker: AAPL
Objective: Conduct structured multi-agent investment research for AAPL.
Step 1: news_analysis -> news_agent
Step 2: market_analysis -> market_agent
Step 3: insider_analysis -> insider_agent
Step 4: risk_analysis -> synthesis_agent
Step 5: catalyst_analysis -> synthesis_agent
Step 6: final_synthesis -> synthesis_agent
Final synthesis
Risks:
- Disposed insider transactions outnumber acquired transactions in the available Form 4 data.
Catalysts:
- Recent analyst or investor commentary may influence market expectations.
- Average daily market return over the available period is positive.
- The available Form 4 data includes insider acquisition transactions.


## 4. Step 2: Turn the Analysis into a Draft

The specialist results become a written draft. Every figure comes directly from the agents'
output, so the evaluator can check it against the raw evidence. The draft also includes the
`Financials & Revenue Performance` section that the optimizer requires, stating that revenue
figures are not in the dataset.

In [4]:
initial_draft = build_research_draft(orchestration)
print(initial_draft)

# Investment Research Draft: AAPL

## Market Performance
- Trading days analyzed: 22 (2026-09-01 to 2026-10-01)
- Latest close: $329.61 on volume of 24,186,647 shares
- Average daily return: 0.07%
- Average daily volume: 41,976,752 shares

## Financials & Revenue Performance
Quarterly Revenue: Exact quarterly revenue dollar figures were not recorded in the primary evidence dataset for this reporting period.

## Insider Activity (SEC Form 4)
- Transactions: 57 by 8 insiders (2026-03-15 to 2026-09-22)
- Acquisitions: 10, dispositions: 47
- Total transaction value: $169,473,295.66

## News Coverage
- Articles analyzed: 2
- Categories: analyst_investor (1), product_service (1)
- Apple: Why I'm Betting Against This Valuable Company (Rating Downgrade) (NASDAQ:AAPL) [analyst_investor]: Apple Inc. has been downgraded to Sell from Hold due to concerns regarding its valuation, specifically trading at a high forward P/E ratio of 38x. This downgrade may indicate potential challenges for the compan

In [5]:
# Tests: the draft reports the agents' numbers and has no missing values.
market = orchestration["specialist_results"]["market_agent"]
insider = orchestration["specialist_results"]["insider_agent"]
news = orchestration["specialist_results"]["news_agent"]

assert f"${market['latest_close']:,.2f}" in initial_draft
assert f"Transactions: {insider['transaction_count']}" in initial_draft
assert f"Articles analyzed: {news['article_count']}" in initial_draft

for risk in orchestration["final_synthesis"]["risks"]:
    assert risk in initial_draft

for section in ["## Market Performance", "## Financials & Revenue Performance",
                "## Insider Activity", "## News Coverage", "## Risks", "## Catalysts"]:
    assert section in initial_draft, section

assert "nan" not in initial_draft.lower().split()

# A ticker with no data still produces a draft instead of an error.
empty_draft = build_research_draft(orchestrate_research("NO_SUCH_TICKER", PROJECT_ROOT))
assert "No market data found" in empty_draft

print("Draft tests passed.")

Draft tests passed.


## 5. Step 3: Collect the Raw Evidence

The evaluator compares the draft to the daily evidence (notebook 02) and the processed news
(notebook 03).

In [6]:
raw_evidence = build_raw_evidence(TICKER, PROJECT_ROOT)
print(f"Raw evidence for {TICKER}: {len(raw_evidence):,} characters")
print(raw_evidence[:1000])

Raw evidence for AAPL: 26,470 characters

=== DAILY MARKET & SEC FORM 4 EVIDENCE (AAPL) ===
                     date ticker       open       high        low      close     volume  daily_return  volume_change  news_count                                                                                                                                                                      news_titles                                                                                                                                                                                                                                      news_urls  insider_transaction_count  insider_total_value                                   insider_names
2026-03-15 00:00:00+00:00   AAPL        NaN        NaN        NaN        NaN        NaN           NaN            NaN           0                                                                                                                                           

## 6. Step 4: Evaluate and Refine

`run_full_research()` runs Steps 1-3 and passes the draft to the Evaluator-Optimizer workflow
from notebook 05. The evaluator scores the draft; if it does not pass (overall and factual
grounding both 8 or higher), the optimizer revises it, for up to three rounds. The final score
is saved to memory.

In [7]:
research = run_full_research(
    TICKER,
    PROJECT_ROOT,
    max_iterations=3,
    memory_store_path=MEMORY_PATH,
)

[AAPL] Loaded Memory Context:
=== HISTORICAL MEMORY FOR AAPL ===
Run #1 [2026-10-05 15:20:49] - Final Quality Score: 8/10
  Summary: Final Score: 8/10 after 5 iterations. Evaluator Notes: ...
Run #2 [2026-10-05 15:21:28] - Final Quality Score: 8/10
  Summary: Final Score: 8/10 after 3 iterations. Evaluator Notes: ...
Run #3 [2026-10-05 15:21:59] - Final Quality Score: 8/10
  Summary: Final Score: 8/10 after 2 iterations. Evaluator Notes: ...
--------------------------------------------------
[AAPL] --- Starting Iteration 1/3 ---
[AAPL] Iteration 1 Score: 4/10 | Passed: False | Failure Reason: Hard gate failure: Missing exact mandatory section headers: ['## Market Metrics', '## SEC Form 4 Insider Activity', '## Financial News Analysis'] | EVERY individual bullet point containing factual metrics MUST include an inline source tag (e.g., [Source: Alpha Vantage], [Source: SEC Form 4], [Source: Marketaux]). Missing on: ['- Trading days analyzed: 22 (2026-09-01 to 2026-10-01)', '- Latest clos

In [8]:
history = pd.DataFrame(
    [
        {
            "iteration": step["iteration"],
            "overall_score": step["overall_score"],
            "passed": step["passed"],
            "factual_grounding": step["rubric"].get("factual_grounding_score"),
            "completeness": step["rubric"].get("completeness_score"),
            "clarity_and_structure": (
                step["rubric"].get("clarity_and_structure_score") 
                or step["rubric"].get("clarity_and_structure")
                or step["rubric"].get("logical_coherence_score")
            ),
        }
        for step in research["history"]
    ]
)
history

,iteration,overall_score,passed,factual_grounding,completeness,clarity_and_structure
0,1,4,False,4,4,4
1,2,8,True,8,10,10


In [9]:
final_eval = research["final_evaluation"]

print(f"Final score: {final_eval.overall_score}/10")
print(f"Passed: {final_eval.passed}")
print(f"Iterations: {len(research['history'])}")
print("\n--- FINAL REPORT ---\n")
print(research["final_report"])

Final score: 8/10
Passed: True
Iterations: 2

--- FINAL REPORT ---

# Investment Research Draft: AAPL

## Market Metrics
- Trading Days Analyzed: 22 (2026-09-01 to 2026-10-01) [Source: Alpha Vantage]
- Latest Close Price: $329.61 on volume of 24,186,647 shares [Source: Alpha Vantage]
- Average Daily Return: 0.07% [Source: Alpha Vantage]
- Average Daily Volume: 41,976,752 shares [Source: Alpha Vantage]

## Financials & Revenue Performance
- Quarterly Revenue: Exact quarterly revenue dollar figures were not recorded in the primary evidence dataset for this reporting period [Source: Marketaux].

## SEC Form 4 Insider Activity
- Transactions: 57 by 8 insiders (2026-03-15 to 2026-09-22) [Source: SEC Form 4]
- Acquisitions: 10, dispositions: 47 [Source: SEC Form 4]
- Total Transaction Value: $169,473,295.66 [Source: SEC Form 4]

## Financial News Analysis
- Articles Analyzed: 2 [Source: Marketaux]
- Categories: analyst_investor (1), product_service (1) [Source: Marketaux]
- Apple: Why I'm Be

## 7. Step 5: Memory Across Runs

Each run's score and evaluator notes are saved for the ticker. The next run for the same ticker
loads this history and passes it to the optimizer.

In [10]:
memory = ResearchMemoryStore(filepath=str(MEMORY_PATH))
print(memory.get_ticker_memory(TICKER))

=== HISTORICAL MEMORY FOR AAPL ===
Run #1 [2026-10-05 15:21:28] - Final Quality Score: 8/10
  Summary: Final Score: 8/10 after 3 iterations. Evaluator Notes: ...
Run #2 [2026-10-05 15:21:59] - Final Quality Score: 8/10
  Summary: Final Score: 8/10 after 2 iterations. Evaluator Notes: ...
Run #3 [2026-10-05 15:22:34] - Final Quality Score: 8/10
  Summary: Final Score: 8/10 after 2 iterations. Evaluator Notes: ...


## 8. Summary

| Step | Component | Notebook |
|---|---|---|
| Plan research steps | Planner agent | 04 |
| Route steps to specialists | Router agent | 04 |
| Analyze news, market, and insider data | Specialist agents | 04 |
| Combine into risks and catalysts | Synthesis agent | 04 |
| Write the draft | `build_research_draft()` | 06 |
| Score the draft | Evaluator | 05 |
| Revise using feedback | Optimizer | 05 |
| Save and reuse run history | Memory | 05 |